# Lending Club -- raw to interim

**What this notebook does:** loads the raw Lending Club file exactly as
published, looks at what's actually in it (`loan_status`, vintage-by-year bad
rate), and materializes the result as a small set of tables in a persistent
DuckDB file. This is the ingestion step -- it does not clean, impute,
transform, encode, or select features. That happens later, in
`notebooks/03_data_cleaning/01_cleaning_and_feature_prep.ipynb`, which reads the
interim file this notebook produces and writes the final modeling table to
`data/03_processed/`.

Keeping ingestion separate from cleaning means every EDA notebook under
`notebooks/02_eda/` can be re-run against a stable, unmodified interim snapshot
regardless of how the cleaning notebook's decisions evolve later.

**Stages, matching the folder structure:**

| Folder | Contents |
|---|---|
| `data/01_raw/` | untouched source file(s) as published by Lending Club |
| `data/02_interim/` | this notebook's output -- a persistent DuckDB file with `raw_mat`, `matured`, `windowed` tables, queried read-only by every EDA notebook |
| `data/03_processed/` | produced later, by the cleaning notebook, not this one |

**Scope: accepted loans only, not the rejected-applications file.** Lending Club also publishes a separate, much larger file of *rejected* applications (people who applied but never got a loan). It's deliberately not loaded here, for a concrete reason: the rejected file has no `loan_status`, no `int_rate`, no repayment history -- nothing that could ever become `is_bad`, because a rejected application was never originated and so has no outcome to observe. It can't be added to `windowed` or used to train or evaluate a PD model the way this notebook's data is used.

What it's actually useful for is a different question entirely --
*reject inference*: checking whether the population Lending Club approved looks different from the population that applied, which matters for understanding sample-selection bias in this entire dataset (every EDA notebook here only ever sees people who were already approved). That's a real gap worth closing, but it's a separate analysis with its own ingestion notebook and its own EDA lane -- not something to fold into this notebook's accepted-loan pipeline. Flagged here as a scope decision, not an oversight: **not yet built**, worth adding as a follow-up once the rejected-applications file is on hand.

**Cell index:**

| # | What it does | What to expect |
|---|---|---|
| 1 | Connect, load the raw file, materialize `raw_mat` | row count of the raw file |
| 2 | Look at `loan_status` as it actually appears in the data | a breakdown table -- which statuses exist, how many rows each |
| 3 | Define "matured" (finished-outcome) loans and the `is_bad` target, build the `matured` table | matured row count and overall bad rate |
| 4 | Bad rate by origination year -- deciding which years are usable | a year-by-year table used to justify the modeling window |
| 5 | Apply the 2013-2017 window, build the `windowed` table | final row count going into every downstream notebook |
| 6 | Summary -- what was produced and what to run next | a recap and a pointer to the cleaning notebook |

**Final output of this notebook:** `data/02_interim/lendingclub.duckdb`,
containing `raw_mat` (all raw rows, typed as text), `matured` (finished-outcome
loans with `is_bad` attached), and `windowed` (the 2013-2017 modeling
population) -- the single source every other Lending Club notebook reads
from.

## Cell 1 -- connect and materialize the raw file

**What / why:** DuckDB can query a gzipped CSV directly through a `VIEW`, but
querying that view repeatedly re-decompresses the whole 392MB gzip every
single time -- expensive, and easy to hit by accident once several notebooks
are querying the same data. Materializing it once into a real table
(`raw_mat`) up front means every later cell -- and every other notebook in
this repo -- pays that decompression cost exactly once, not once per query.

Everything is loaded as text (`all_varchar=True`) rather than DuckDB's
auto-inferred types, because Lending Club's raw export mixes numeric-looking
columns with stray text values (e.g. "n/a") that would otherwise make type
inference silently drop or corrupt rows. Casting happens deliberately later,
per column, in the EDA notebooks -- not implicitly here.

**How:** connect to a persistent DuckDB file (not `:memory:`) so the tables
survive after this notebook finishes running; create a `VIEW` over
`read_csv(...)`, then immediately `CREATE OR REPLACE TABLE raw_mat AS SELECT
* FROM raw` to force the one-time materialization.

**Expect:** a single printed row count for the raw file -- on the order of
2.26 million loans, matching Lending Club's full 2007-2018Q4 accepted-loan
export.

In [1]:
import sys, os, duckdb
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import create_fresh

RAW_FILE = "../../data/01_raw/accepted_2007_to_2018Q4.csv.gz"
con, ASSETS_TABLES, ASSETS_PLOTS = create_fresh()  # deletes any old interim file, starts clean

con.sql(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT * FROM read_csv('{RAW_FILE}', header=True, delim=',',
                            ignore_errors=True, all_varchar=True)
""")
con.sql("CREATE OR REPLACE TABLE raw_mat AS SELECT * FROM raw")

n_raw = con.sql("SELECT count(*) FROM raw_mat").fetchone()[0]
print(f"raw file: {n_raw:,} rows")


raw file: 2,260,701 rows


**What the output shows:** the raw file has 2,260,701
rows -- this is the entire published Lending Club accepted-loan history,
before any filtering. It's now sitting in `raw_mat` inside the persistent
DuckDB file, so nothing downstream needs to touch the gzip again.

**Next:** this is the *whole* population, including loans that haven't
reached a final outcome yet. Before anything else, checking what `loan_status`
actually contains -- that's what determines which rows can be used at all.

## Cell 2 -- what does loan_status actually contain?

**What / why:** before deciding anything, looking directly at the raw
`loan_status` values rather than assuming what they are. This drives every
downstream decision about which loans are usable: some statuses represent a
finished loan (paid off or defaulted), others represent a loan still in
progress whose eventual outcome isn't known yet. Getting this list right
before building anything else avoids silently mislabeling still-open loans as
good or bad.

**How:** a simple `GROUP BY loan_status` count, ordered by frequency.

**Expect:** a handful of distinct statuses, dominated by `Fully Paid`,
`Current`, and `Charged Off`, plus smaller categories for `Late`, `In Grace
Period`, `Default`, and two "does not meet the credit policy" legacy
variants.

In [2]:
print("loan_status breakdown:")
status_breakdown = con.sql(
    "SELECT loan_status, count(*) AS n FROM raw_mat GROUP BY 1 ORDER BY 2 DESC"
).df()
print(status_breakdown.to_string(index=False))
status_breakdown.to_csv(os.path.join(ASSETS_TABLES, "ing01_status_breakdown.csv"), index=False)


loan_status breakdown:


                                        loan_status       n
                                         Fully Paid 1076751
                                            Current  878317
                                        Charged Off  268559
                                 Late (31-120 days)   21467
                                    In Grace Period    8436
                                  Late (16-30 days)    4349
 Does not meet the credit policy. Status:Fully Paid    1988
Does not meet the credit policy. Status:Charged Off     761
                                            Default      40
                                                NaN      33


**What the output shows:**
```
loan_status breakdown:
                                        loan_status       n
                                         Fully Paid 1076751
                                            Current  878317
                                        Charged Off  268559
                                 Late (31-120 days)   21467
                                    In Grace Period    8436
                                  Late (16-30 days)    4349
 Does not meet the credit policy. Status:Fully Paid    1988
Does not meet the credit policy. Status:Charged Off     761
                                            Default      40
                                                NaN      33
```
Only some of these statuses mean the loan is actually finished. `Current`,
`Late (...)`, and `In Grace Period` loans are still open -- their eventual
outcome isn't known yet, so they can't be used to train or evaluate a model
(explored further in `notebooks/02_eda/07_target_outcome_objective.ipynb`).
`Fully Paid`, `Charged Off`, `Default`, and the two "does not meet the credit
policy" variants all represent a *final* outcome.

In [3]:
print("the 33 rows where loan_status is NaN -- what do they actually contain?")
sample = con.sql("SELECT * FROM raw_mat WHERE loan_status IS NULL LIMIT 5").df()
non_empty_cols = [c for c in sample.columns if sample[c].notna().any()]
print(f"columns with anything in them, out of {len(sample.columns)} total: {non_empty_cols}")
print(sample[non_empty_cols].to_string(index=False))
sample[non_empty_cols].to_csv(os.path.join(ASSETS_TABLES, "ing01_null_status_rows.csv"), index=False)


the 33 rows where loan_status is NaN -- what do they actually contain?
columns with anything in them, out of 151 total: ['id']
                                              id
Total amount funded in policy code 1: 6417608175
Total amount funded in policy code 2: 1944088810
Total amount funded in policy code 1: 1741781700
 Total amount funded in policy code 2: 564202131
Total amount funded in policy code 1: 1791201400


**What the 33 `NaN` rows actually are:** the query below shows it directly -- not real loan records. Every field is empty except `id`, which holds text like `"Total amount funded in policy code 1: 6417608175"` -- these are summary/footer lines Lending Club appended to the bottom of the raw CSV export, not loans. `read_csv(..., ignore_errors=True)` in cell 1 let them through as 33 all-null rows instead of rejecting them outright. They fall out of `matured` automatically in cell 3 (a `NULL loan_status` can't match any status in `MATURED_STATUSES`), but that was previously an *implicit* side effect rather than a deliberate, documented filter -- worth calling out explicitly here so nobody downstream mistakes 33 rows of real data for silently vanishing.

**Why loans still in progress (`Current`, `Late (...)`, `In Grace Period`) can't go into the model -- and what they're for instead:** these are real, active loans; they're excluded from `matured` for one specific reason: `is_bad` isn't yet *known* for them. A loan that's still being paid on time today could still default next year, so labeling it "good" now would be guessing, not measuring -- training on that guess would teach the model a wrong answer with false confidence. That's the whole reason this notebook draws a hard line between "matured" (usable for training/evaluation, because the outcome already happened) and everything else.

That doesn't make the open loans useless -- just not usable *as training labels*. Once Phase 1 has a trained model, the natural use for this excluded population is **scoring, not training**: running the model on `Current`/`Late`/`In Grace Period` loans to get a live risk estimate for loans that haven't resolved yet (a model only needs features to score a loan, not a known outcome). They're also the natural population for **ongoing monitoring** -- watching whether today's `Late`/`In Grace Period` loans go on to cure or default, which is exactly how this whole labeling scheme eventually gets re-validated against reality. The size of this excluded population and a closer look at the `Default` vs. `Charged Off` boundary are both explored in `notebooks/02_eda/07_target_outcome_objective.ipynb`, cell 2.

**Next:** defining exactly which statuses count as "matured" (finished) and
which of those are "bad" (defaulted), then building a table restricted to
only the matured population.

## Cell 3 -- define matured loans and the is_bad target

**What / why:** this is where the modeling target gets defined. `is_bad = 1`
for `Charged Off`, `Default`, and the "does not meet the credit policy:
Charged Off" variant; `is_bad = 0` for `Fully Paid` and its "does not meet the
credit policy" variant. Everything downstream -- every IV number, every WOE
table, the baseline logistic regression -- is relative to this exact
definition, so it's built once, explicitly, here, rather than being
re-derived differently in different notebooks.

**How:** filter `raw_mat` down to only the matured statuses from cell 2,
attaching `is_bad` via a `CASE WHEN` over the bad-status list.

**Expect:** roughly 60% of the raw population is matured (the rest is still
open), and a bad rate somewhere in the high teens to low twenties.

In [4]:
MATURED_STATUSES = (
    "Fully Paid", "Charged Off", "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Does not meet the credit policy. Status:Fully Paid",
)
BAD_STATUSES = (
    "Charged Off", "Default",
    "Does not meet the credit policy. Status:Charged Off",
)
matured_list = ", ".join(f"'{s}'" for s in MATURED_STATUSES)
bad_list = ", ".join(f"'{s}'" for s in BAD_STATUSES)

con.sql(f"""
    CREATE OR REPLACE TABLE matured AS
    SELECT *, CASE WHEN loan_status IN ({bad_list}) THEN 1 ELSE 0 END AS is_bad
    FROM raw_mat
    WHERE loan_status IN ({matured_list})
""")
n_matured, n_bad = con.sql("SELECT count(*), sum(is_bad) FROM matured").fetchone()
print(f"matured (finished) loans: {n_matured:,} of {n_raw:,}")
print(f"bad rate among matured loans: {n_bad / n_matured:.1%}")


matured (finished) loans: 1,348,099 of 2,260,701
bad rate among matured loans: 20.0%


**What the output shows:** 1,348,099 of the
2,260,701 raw loans (59.6%) have a final
outcome; the rest are still open and excluded. Among the matured population,
the overall bad rate is 20.0%. This is the headline
number the rest of the repo is built against -- it's what every "bad rate by
X" table downstream is being compared to.

**Next:** the matured population spans 2007 through 2018. Before using all of
it, checking whether every year is actually reliable to model on -- some
early years have too little volume, and the most recent year is
right-censored (its bad loans haven't all had time to default yet).

## Cell 4 -- bad rate by origination year

**What / why:** deciding the modeling window requires actually looking at
volume and bad rate per year, not assuming the full history is usable.
Vintage effects matter here for two different reasons: 2007-2012 was Lending
Club's early, low-volume period, and the most recent vintage in the raw file
is subject to right-censoring -- a loan that just originated hasn't had time
to default yet, so only its fastest-resolving outcomes (quick payoffs, quick
early defaults) show up as "matured," making that year look artificially
safe.

**How:** group `matured` by origination year (parsed from `issue_d`) and
compute row count and bad rate per year.

**Expect:** low volume in the earliest years, a stable bad rate through the
middle years, and a noticeably lower bad rate in the most recent year than
the trend would suggest -- the right-censoring signature.

In [5]:
print("bad rate by origination year:")
by_year = con.sql("""
    SELECT substr(issue_d, -4) AS year, count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM matured GROUP BY 1 ORDER BY 1
""").df()
print(by_year.to_string(index=False))
by_year.to_csv(os.path.join(ASSETS_TABLES, "ing01_by_year.csv"), index=False)


bad rate by origination year:
year      n  bad_rate
2007    603     0.262
2008   2393     0.207
2009   5281     0.137
2010  12537     0.140
2011  21721     0.152
2012  53367     0.162
2013 134804     0.156
2014 223103     0.184
2015 375546     0.202
2016 293105     0.233
2017 169321     0.231
2018  56318     0.158


**What the output shows:**
```
bad rate by origination year:
year      n  bad_rate
2007    603     0.262
2008   2393     0.207
2009   5281     0.137
2010  12537     0.140
2011  21721     0.152
2012  53367     0.162
2013 134804     0.156
2014 223103     0.184
2015 375546     0.202
2016 293105     0.233
2017 169321     0.231
2018  56318     0.158
```
This confirms both concerns going in: the earliest years have too few loans
per year to trust, and the most recent year's bad rate sits well below the
2015-2017 trend -- the right-censoring effect, not a genuine improvement in
credit quality. Explored further in
`notebooks/02_eda/06_temporal_sequential_spatial.ipynb`.

**Next:** applying a window that excludes both the unreliable early years and
the right-censored most recent year, and materializing that as the `windowed`
table -- the actual population every EDA and modeling notebook uses.

## Cell 5 -- apply the modeling window

**What / why:** based on cell 4, restricting to 2013-2017: recent enough to
be representative of current underwriting practice, but old enough that every
loan has had the full term to reach a final outcome, avoiding the
right-censoring bias seen in the most recent vintage.

**How:** filter `matured` to `issue_d` years between 2013 and 2017 inclusive,
materialize as `windowed`.

**Expect:** a row count noticeably smaller than the full matured population
(roughly half), since this drops both the 2007-2012 tail and 2018.

In [6]:
VINTAGE_START, VINTAGE_END = 2013, 2017

con.sql(f"""
    CREATE OR REPLACE TABLE windowed AS
    SELECT * FROM matured
    WHERE CAST(substr(issue_d, -4) AS INT) BETWEEN {VINTAGE_START} AND {VINTAGE_END}
""")
n_windowed = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"rows in {VINTAGE_START}-{VINTAGE_END} window: {n_windowed:,}")


rows in 2013-2017 window: 1,195,879


**What the output shows:** the 2013-
2017 window keeps 1,195,879 of the
1,348,099 matured loans (88.7%) --
this `windowed` table is the actual modeling population every EDA notebook
under `notebooks/02_eda/` and the cleaning notebook query from here forward.

**Next:** the interim DuckDB file is complete. A short summary of what was
produced and what to run next.

## Cell 6 -- summary and next step

**What / why:** closing out with an explicit statement of what this notebook
produced, so anyone picking up the repo doesn't have to re-read every cell to
know what's available downstream.

**How:** print the output file path and each table's row count -- facts only, no restating in prose what the markdown cells around this one already say (that convention -- explanation lives in markdown, code cells print results and nothing else -- applies to every cell in this repo, not just this one).

**Expect:** a short recap, and a pointer to the next notebook to run.

In [7]:
n_tables = {}
for tbl in ["raw_mat", "matured", "windowed"]:
    n_tables[tbl] = con.sql(f"SELECT count(*) FROM {tbl}").fetchone()[0]

print(f"file: {os.path.abspath('../../data/02_interim/lendingclub.duckdb')}")
for tbl, n in n_tables.items():
    print(f"{tbl}: {n:,} rows")
con.close()


file: /tmp/lc_eda/lendingclub/data/02_interim/lendingclub.duckdb
raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows


**What the output shows:** confirmation that
`data/02_interim/lendingclub.duckdb` now holds `raw_mat`
(2,260,701 rows), `matured` (1,348,099 rows), and `windowed`
(1,195,879 rows, 20.0% bad rate on the
matured population) -- this file is the single source of truth every other
Lending Club notebook in this repo reads from.

**Next:** this notebook's job is done. From here, either explore the data
further via `notebooks/02_eda/01_data_understanding_structural_profiling.ipynb`
onward, or go straight to `notebooks/03_data_cleaning/01_cleaning_and_feature_prep.ipynb`
to produce the final modeling table.